In [2]:
import pandas as pd

# Charger le fichier
df = pd.read_csv(r"C:/Users/PC/DATA/raw/fusion_polluants_2026(in).csv", sep=";", encoding="utf-8", dtype=str)

# Colonnes à garder
colonnes_utiles = [
    "Date de début",
    "Date de fin",
    "Polluant",
    "valeur",
    "unité de mesure",
    "code site",
    "nom site",
    "type d'implantation",
    "type d'influence",
    "validité"
]

# Sélection
df_clean = df[colonnes_utiles].copy()

# Conversion des types
df_clean["Date de début"] = pd.to_datetime(df_clean["Date de début"], errors="coerce")
df_clean["Date de fin"] = pd.to_datetime(df_clean["Date de fin"], errors="coerce")
df_clean["valeur"] = pd.to_numeric(df_clean["valeur"], errors="coerce")
df_clean["validité"] = pd.to_numeric(df_clean["validité"], errors="coerce")

# Nettoyage léger (sans supprimer trop de données)
df_clean = df_clean[df_clean["valeur"].notna()]
df_clean = df_clean[df_clean["valeur"] >= 0]

# Export
df_clean.to_csv("Pollution.csv", sep=";", index=False, encoding="utf-8-sig")

print("✅ Fichier nettoyé exporté : Pollution.csv")

C:\Users\PC\AppData\Local\Temp\ipykernel_7004\3920979255.py:24: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_clean["Date de début"] = pd.to_datetime(df_clean["Date de début"], errors="coerce")
C:\Users\PC\AppData\Local\Temp\ipykernel_7004\3920979255.py:25: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_clean["Date de fin"] = pd.to_datetime(df_clean["Date de fin"], errors="coerce")


✅ Fichier nettoyé exporté : Pollution.csv


In [3]:
df_clean

,Date de début,Date de fin,Polluant,valeur,unité de mesure,code site,nom site,type d'implantation,type d'influence,validité
0,2026-03-15 00:00:00,2026-03-15 01:00:00,NO,0.2,µg-m3,FR01011,Metz-Centre,Urbaine,Fond,1
1,2026-03-15 00:00:00,2026-03-15 01:00:00,O3,7.2,µg-m3,FR38014,Ligne Paradis,Périurbaine,Fond,4
2,2026-03-15 00:00:00,2026-03-15 01:00:00,PM10,5.5,µg-m3,FR09203,La Pallice3 LR,Urbaine,Industrielle,1
3,2026-03-15 00:00:00,2026-03-15 01:00:00,PM2.5,7.7,µg-m3,FR09203,La Pallice3 LR,Urbaine,Industrielle,1
4,2026-03-15 00:00:00,2026-03-15 01:00:00,NO2,16.7,µg-m3,FR38014,Ligne Paradis,Périurbaine,Fond,1
...,...,...,...,...,...,...,...,...,...,...
736333,2026-03-30 14:00:00,2026-03-30 15:00:00,O3,37.7,µg-m3,FR38014,Ligne Paradis,Périurbaine,Fond,4
736334,2026-03-30 14:00:00,2026-03-30 15:00:00,SO2,2.7,µg-m3,FR38018,Centre Penitencier,Périurbaine,Industrielle,1
736335,2026-03-30 14:00:00,2026-03-30 15:00:00,NO,8.3,µg-m3,FR38098,Sarda Garriga,Urbaine,Trafic,1
736340,2026-03-30 14:00:00,2026-03-30 15:00:00,NO2,7.4,µg-m3,FR38098,Sarda Garriga,Urbaine,Trafic,1


In [4]:
polluants = ["NO2", "PM10", "PM2.5"]
df_clean = df_clean[df_clean["Polluant"].isin(polluants)]

In [5]:
df_pivot = df_clean.pivot_table(
    index=["Date de début", "code site"],
    columns="Polluant",
    values="valeur",
    aggfunc="mean"
).reset_index()

In [6]:
df_pivot.columns.name = None

In [7]:
df_pivot = df_pivot.dropna()

In [8]:
df_pivot

,Date de début,code site,NO2,PM10,PM2.5
0,2026-03-15 00:00:00,FR01011,7.0,4.8,1.0
5,2026-03-15 00:00:00,FR01020,28.2,13.5,11.0
11,2026-03-15 00:00:00,FR02022,2.1,7.9,1.9
12,2026-03-15 00:00:00,FR02041,2.1,3.8,1.9
14,2026-03-15 00:00:00,FR03006,7.3,5.6,0.9
...,...,...,...,...,...
150609,2026-03-30 10:00:00,FR82040,6.4,4.7,3.6
150614,2026-03-30 11:00:00,FR38001,5.2,8.7,3.9
150626,2026-03-30 12:00:00,FR38001,4.3,7.4,3.4
150636,2026-03-30 13:00:00,FR38001,3.9,9.1,4.0


In [9]:
station = pd.read_excel(r"C:/Users/PC/DATA/raw/fr-2025-d-lcsqa-ineris-20251209.xls")

In [11]:
station_clean = station[[
    "LocalId",   # ou équivalent
    "Latitude",
    "Longitude"
]]

In [13]:
station_clean

,LocalId,Latitude,Longitude
0,STA-FR19012,48.386180,-4.486600
1,STA-FR34051,46.798280,1.693139
2,STA-FR12047,43.803140,1.595475
3,STA-FR16034,48.590430,7.744983
4,STA-FR30026,48.584824,6.483900
...,...,...,...
863,STA-FR33102,45.596670,5.918611
864,STA-FR04333,48.948124,2.248817
865,STA-FR05088,49.749943,0.415019
866,STA-FR08018,43.692600,3.800210


In [14]:
df_pivot["code site"].unique()[:10]
station_clean["LocalId"].unique()[:10]

array(['STA-FR19012', 'STA-FR34051', 'STA-FR12047', 'STA-FR16034',
       'STA-FR30026', 'STA-FR12031', 'STA-FR18055', 'STA-FR82512',
       'STA-FR35012', 'STA-FR31019'], dtype=object)

In [15]:
df_pivot["code site clean"] = "STA-" + df_pivot["code site"]

In [16]:
df_pollution = df_pivot.merge(
    station_clean,
    left_on="code site clean",
    right_on="LocalId",
    how="left"
)

In [17]:
df_pollution

,Date de début,code site,NO2,PM10,PM2.5,code site clean,LocalId,Latitude,Longitude
0,2026-03-15 00:00:00,FR01011,7.0,4.8,1.0,STA-FR01011,STA-FR01011,49.119442,6.180833
1,2026-03-15 00:00:00,FR01020,28.2,13.5,11.0,STA-FR01020,STA-FR01020,49.358337,6.156942
2,2026-03-15 00:00:00,FR02022,2.1,7.9,1.9,STA-FR02022,STA-FR02022,43.675114,4.629210
3,2026-03-15 00:00:00,FR02041,2.1,3.8,1.9,STA-FR02041,STA-FR02041,43.639004,5.101097
4,2026-03-15 00:00:00,FR03006,7.3,5.6,0.9,STA-FR03006,STA-FR03006,43.276450,5.397360
...,...,...,...,...,...,...,...,...,...
68435,2026-03-30 10:00:00,FR82040,6.4,4.7,3.6,STA-FR82040,STA-FR82040,47.096720,5.496389
68436,2026-03-30 11:00:00,FR38001,5.2,8.7,3.9,STA-FR38001,STA-FR38001,-20.889893,55.468840
68437,2026-03-30 12:00:00,FR38001,4.3,7.4,3.4,STA-FR38001,STA-FR38001,-20.889893,55.468840
68438,2026-03-30 13:00:00,FR38001,3.9,9.1,4.0,STA-FR38001,STA-FR38001,-20.889893,55.468840


In [18]:
df_pollution = df_pollution[
    (df_pollution["Latitude"] >= 41) &
    (df_pollution["Latitude"] <= 51) &
    (df_pollution["Longitude"] >= -5) &
    (df_pollution["Longitude"] <= 10)
]

In [19]:
df_pollution

,Date de début,code site,NO2,PM10,PM2.5,code site clean,LocalId,Latitude,Longitude
0,2026-03-15 00:00:00,FR01011,7.0,4.8,1.0,STA-FR01011,STA-FR01011,49.119442,6.180833
1,2026-03-15 00:00:00,FR01020,28.2,13.5,11.0,STA-FR01020,STA-FR01020,49.358337,6.156942
2,2026-03-15 00:00:00,FR02022,2.1,7.9,1.9,STA-FR02022,STA-FR02022,43.675114,4.629210
3,2026-03-15 00:00:00,FR02041,2.1,3.8,1.9,STA-FR02041,STA-FR02041,43.639004,5.101097
4,2026-03-15 00:00:00,FR03006,7.3,5.6,0.9,STA-FR03006,STA-FR03006,43.276450,5.397360
...,...,...,...,...,...,...,...,...,...
68431,2026-03-30 10:00:00,FR41017,6.2,15.6,5.4,STA-FR41017,STA-FR41017,42.671333,9.434639
68432,2026-03-30 10:00:00,FR42010,12.1,2.9,1.7,STA-FR42010,STA-FR42010,48.572807,7.767833
68433,2026-03-30 10:00:00,FR50060,8.9,11.3,5.2,STA-FR50060,STA-FR50060,44.012856,1.375305
68434,2026-03-30 10:00:00,FR82010,5.2,2.7,2.1,STA-FR82010,STA-FR82010,47.510307,6.794000


In [23]:
df_pollution = df_pollution.drop(columns=[
    "code site clean",
    "LocalId"
])

In [24]:
df_pollution["Latitude"].isna().sum()

0

In [25]:
df_pollution

,Date de début,code site,NO2,PM10,PM2.5,Latitude,Longitude
0,2026-03-15 00:00:00,FR01011,7.0,4.8,1.0,49.119442,6.180833
1,2026-03-15 00:00:00,FR01020,28.2,13.5,11.0,49.358337,6.156942
2,2026-03-15 00:00:00,FR02022,2.1,7.9,1.9,43.675114,4.629210
3,2026-03-15 00:00:00,FR02041,2.1,3.8,1.9,43.639004,5.101097
4,2026-03-15 00:00:00,FR03006,7.3,5.6,0.9,43.276450,5.397360
...,...,...,...,...,...,...,...
68431,2026-03-30 10:00:00,FR41017,6.2,15.6,5.4,42.671333,9.434639
68432,2026-03-30 10:00:00,FR42010,12.1,2.9,1.7,48.572807,7.767833
68433,2026-03-30 10:00:00,FR50060,8.9,11.3,5.2,44.012856,1.375305
68434,2026-03-30 10:00:00,FR82010,5.2,2.7,2.1,47.510307,6.794000


In [27]:
df_pollution.to_csv("pollution_clean.csv", index=False)